In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. Configuration
# ============================================================

def find_project_root():
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists() and (path / "Result").exists():
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )

PROJECT_ROOT = find_project_root()

RESULT_ROOT = (
    PROJECT_ROOT
    / "Result"
    / "04_Foldwise_Correlation_Clustering_Champion_Selection_Rho_Sensitivity"
)

OUT_DIR = (
    PROJECT_ROOT
    / "Result"
    / "05_Stable_Cluster_Champions_and_Novel_Ratio_Candidates"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMPUTATION_METHODS = ["global_mean", "knn"]
N_OUTER_FOLDS = 5

# Main analysis threshold.
RHO_TAG = "0.90"

# Total method-fold combinations:
# global_mean 5 folds + knn 5 folds = 10.
TOTAL_METHOD_FOLD_COMBOS = len(IMPUTATION_METHODS) * N_OUTER_FOLDS

# Minimum appearance count for candidate features reported in the paper.
MIN_APPEARANCES_FOR_PAPER = 3

# Minimum appearance count for core stable features.
MIN_APPEARANCES_CORE = 6

DPI = 300


# ============================================================
# 1. Utility functions
# ============================================================

def find_rho_result_file(method, fold, rho_tag="0.90"):
    """
    Find the result Excel file for a given method, outer fold, and rho threshold.
    """
    rho_dir = (
        RESULT_ROOT
        / method
        / f"outer_fold_{fold:02d}"
        / f"rho_{rho_tag}"
    )

    if not rho_dir.exists():
        raise FileNotFoundError(f"Directory was not found: {rho_dir}")

    expected_name = (
        f"outer_fold_{fold:02d}_cluster_champions_SHAP_innerCV_rho{rho_tag}.xlsx"
    )
    expected_path = rho_dir / expected_name

    if expected_path.exists():
        return expected_path

    # Fallback: automatic search.
    candidates = [
        p for p in rho_dir.iterdir()
        if (
            p.is_file()
            and p.suffix.lower() == ".xlsx"
            and "cluster_champions" in p.name
            and f"rho{rho_tag}" in p.name
        )
    ]

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No rho={rho_tag} result file was found in: {rho_dir}"
        )

    return candidates[0]


def read_sheet_if_exists(xlsx_path, sheet_candidates):
    """
    Try to read the first available sheet from a list of candidate sheet names.
    """
    xls = pd.ExcelFile(xlsx_path)
    available = xls.sheet_names

    for sheet_name in sheet_candidates:
        if sheet_name in available:
            return pd.read_excel(xlsx_path, sheet_name=sheet_name)

    return pd.DataFrame()


def display_feature_name(name):
    """
    Format feature names for display only.

    The original feature IDs are not modified.
    """
    s = str(name)

    # Keep the current dataset convention as the display convention.
    # Compatibility replacements are retained only for older intermediate outputs.
    s = s.replace("10000*Ga/A1", "10000xGa/Al")
    s = s.replace("10000*Ga/Al", "10000xGa/Al")
    s = s.replace("10000×Ga/Al", "10000xGa/Al")
    s = s.replace("A12O3", "Al2O3")

    # Remove ratio prefixes for clearer display.
    s = s.replace("R_Major_", "")
    s = s.replace("R_Trace_", "")

    return s


def is_constructed_ratio(feature):
    """
    Check whether a feature is a systematically constructed ratio feature.
    """
    s = str(feature)
    return s.startswith("R_Major_") or s.startswith("R_Trace_")


def ratio_group(feature):
    """
    Assign a simple geochemical group to a ratio feature.
    """
    s = str(feature)

    if not is_constructed_ratio(s):
        return "Non-ratio / original feature"

    if s.startswith("R_Major_"):
        return "Major-element ratio"

    if s.startswith("R_Trace_"):
        body = s.replace("R_Trace_", "")

        hfse_keys = ["Nb", "Ta", "Zr", "Hf", "Ti", "Y"]
        ree_keys = [
            "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd",
            "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"
        ]
        lile_keys = ["Rb", "Sr", "Ba", "Cs", "Pb"]
        radioactive_keys = ["Th", "U"]

        parts = re.split(r"[/_+\-*()]+", body)
        parts = [p for p in parts if p]

        has_hfse = any(p in hfse_keys for p in parts)
        has_ree = any(p in ree_keys for p in parts)
        has_lile = any(p in lile_keys for p in parts)
        has_radioactive = any(p in radioactive_keys for p in parts)

        if has_hfse and has_ree:
            return "HFSE-REE ratio"
        if has_hfse and has_radioactive:
            return "HFSE-Th-U ratio"
        if has_hfse:
            return "HFSE ratio"
        if has_ree:
            return "REE ratio"
        if has_lile:
            return "LILE-related ratio"

        return "Trace-element ratio"

    return "Other ratio"


def is_common_classical_ratio(feature):
    """
    Mark commonly used classical indicators.

    This flag is used only to distinguish candidate novel ratios from
    classical or widely used geochemical indicators.
    """
    s = str(feature)

    common_patterns = [
        "A/CNK",
        "A/NK",
        "10000xGa/Al",
        "10000*Ga/Al",
        "10000*Ga/A1",
        "10000×Ga/Al",
        "Zr+Nb+Ce+Y",
        "R_Trace_Sr/Y",
        "R_Trace_Nb/Ta",
        "R_Trace_Zr/Hf",
        "R_Major_K2O/Na2O",
        "R_Major_Fe2O3t/MgO",
    ]

    return any(p == s or p in s for p in common_patterns)


def stability_level(row):
    """
    Assign a stability level according to appearance frequency.
    """
    appearances = row["Appearance_count"]
    method_count = row["Method_count"]

    if appearances >= MIN_APPEARANCES_CORE and method_count == 2:
        return "Core stable feature"

    if appearances >= MIN_APPEARANCES_FOR_PAPER and method_count == 2:
        return "Moderately stable feature"

    if appearances >= MIN_APPEARANCES_FOR_PAPER:
        return "Method-specific stable feature"

    return "Occasional feature"


def safe_mean(series):
    s = pd.to_numeric(series, errors="coerce")
    return float(s.mean()) if s.notna().any() else np.nan


def safe_median(series):
    s = pd.to_numeric(series, errors="coerce")
    return float(s.median()) if s.notna().any() else np.nan


def standardize_ablation_columns(df):
    """
    Standardize ablation column names from different script versions.

    This does not change the workflow. It only makes the summary script
    compatible with equivalent English column names.
    """
    if df.empty:
        return df

    df = df.copy()

    rename_map = {
        "delta_f1_mean_full_minus_drop": "delta_f1_mean(full-drop)",
        "delta_f1_std": "delta_f1_std",
    }

    for old_col, new_col in rename_map.items():
        if old_col in df.columns and new_col not in df.columns:
            df[new_col] = df[old_col]

    if "positive_split_ratio" not in df.columns and "positive_folds" in df.columns:
        df["positive_split_ratio"] = (
            pd.to_numeric(df["positive_folds"], errors="coerce") / N_OUTER_FOLDS
        )

    return df


# ============================================================
# 2. Collect all rho=0.90 ClusterChampion and ablation results
# ============================================================

def collect_all_rho090_results():
    champion_rows = []
    ablation_rows = []
    feature_score_rows = []

    for method in IMPUTATION_METHODS:
        for fold in range(1, N_OUTER_FOLDS + 1):
            xlsx_path = find_rho_result_file(method, fold, RHO_TAG)

            print(f"Reading: {method} fold {fold} -> {xlsx_path}")

            # 1) ClusterChampions.
            champ_df = read_sheet_if_exists(
                xlsx_path,
                ["ClusterChampions"]
            )

            if champ_df.empty:
                print(f"Warning: ClusterChampions sheet was not found: {xlsx_path}")
            else:
                champ_df["Method"] = method
                champ_df["Outer_fold"] = fold
                champ_df["Source_file"] = str(xlsx_path)

                if "champion" not in champ_df.columns:
                    raise ValueError(
                        f"The ClusterChampions sheet has no 'champion' column: {xlsx_path}"
                    )

                champion_rows.append(champ_df)

            # 2) Feature scores.
            feat_score_df = read_sheet_if_exists(
                xlsx_path,
                ["FeatureScores_SHAP_innerCV", "FeatureScores_SHAP_CV"]
            )

            if not feat_score_df.empty:
                feat_score_df["Method"] = method
                feat_score_df["Outer_fold"] = fold
                feat_score_df["Source_file"] = str(xlsx_path)
                feature_score_rows.append(feat_score_df)

            # 3) Ablation results.
            ab_df = read_sheet_if_exists(
                xlsx_path,
                [
                    "Ablation_Stats_innerCV",
                    "Ablation_Champions_innerCV",
                    "Ablation_Champions"
                ]
            )

            if not ab_df.empty:
                # Exclude message-only sheets such as "Ablation not run".
                if "feature" in ab_df.columns:
                    ab_df = standardize_ablation_columns(ab_df)
                    ab_df["Method"] = method
                    ab_df["Outer_fold"] = fold
                    ab_df["Source_file"] = str(xlsx_path)
                    ablation_rows.append(ab_df)

    all_champions = (
        pd.concat(champion_rows, axis=0, ignore_index=True)
        if champion_rows else pd.DataFrame()
    )

    all_feature_scores = (
        pd.concat(feature_score_rows, axis=0, ignore_index=True)
        if feature_score_rows else pd.DataFrame()
    )

    all_ablation = (
        pd.concat(ablation_rows, axis=0, ignore_index=True)
        if ablation_rows else pd.DataFrame()
    )

    return all_champions, all_feature_scores, all_ablation


# ============================================================
# 3. Summarize cluster-champion stability
# ============================================================

def summarize_champion_stability(all_champions):
    df = all_champions.copy()

    df["Feature"] = df["champion"].astype(str)
    df["Display_feature"] = df["Feature"].apply(display_feature_name)
    df["Is_constructed_ratio"] = df["Feature"].apply(is_constructed_ratio)
    df["Ratio_group"] = df["Feature"].apply(ratio_group)
    df["Is_common_classical_ratio"] = df["Feature"].apply(is_common_classical_ratio)
    df["Is_candidate_novel_ratio"] = (
        df["Is_constructed_ratio"] & (~df["Is_common_classical_ratio"])
    )

    method_pivot = (
        df
        .pivot_table(
            index="Feature",
            columns="Method",
            values="Outer_fold",
            aggfunc="count",
            fill_value=0
        )
        .reset_index()
    )

    for method in IMPUTATION_METHODS:
        if method not in method_pivot.columns:
            method_pivot[method] = 0

    method_pivot = method_pivot.rename(
        columns={
            "global_mean": "Global_mean_count",
            "knn": "KNN_count"
        }
    )

    agg = (
        df
        .groupby("Feature", as_index=False)
        .agg(
            Display_feature=("Display_feature", "first"),
            Appearance_count=("Feature", "count"),
            Appearance_rate=("Feature", lambda x: len(x) / TOTAL_METHOD_FOLD_COMBOS),
            Method_count=("Method", lambda x: x.nunique()),
            Methods_present=("Method", lambda x: ",".join(sorted(x.unique()))),
            Folds_present=("Outer_fold", lambda x: ",".join(map(str, sorted(x.unique())))),
            Fold_count=("Outer_fold", lambda x: x.nunique()),

            Mean_cluster_size=("cluster_size", safe_mean),
            Median_cluster_size=("cluster_size", safe_median),

            Mean_champion_score=("champion_score", safe_mean),
            Median_champion_score=("champion_score", safe_median),

            Mean_SHAP_importance=("champion_importance_mean", safe_mean),
            Mean_TopK_frequency=("champion_topk_freq_ratio", safe_mean),

            Is_constructed_ratio=("Is_constructed_ratio", "first"),
            Ratio_group=("Ratio_group", "first"),
            Is_common_classical_ratio=("Is_common_classical_ratio", "first"),
            Is_candidate_novel_ratio=("Is_candidate_novel_ratio", "first"),
        )
    )

    agg = agg.merge(method_pivot, on="Feature", how="left")

    agg["Stability_level"] = agg.apply(stability_level, axis=1)

    agg = agg.sort_values(
        by=[
            "Appearance_count",
            "Method_count",
            "Mean_champion_score",
            "Mean_SHAP_importance"
        ],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    return df, agg


# ============================================================
# 4. Summarize ablation statistics
# ============================================================

def summarize_ablation(all_ablation):
    if all_ablation.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = standardize_ablation_columns(all_ablation)

    df["Feature"] = df["feature"].astype(str)
    df["Display_feature"] = df["Feature"].apply(display_feature_name)

    if "Decision" not in df.columns:
        df["Decision"] = "Not available"

    required_numeric_cols = [
        "delta_f1_mean(full-drop)",
        "delta_f1_std",
        "delta_f1_t_ci95_lower",
        "delta_f1_t_ci95_upper",
        "bootstrap_ci95_lower",
        "bootstrap_ci95_upper",
        "positive_split_ratio",
        "paired_ttest_p",
        "paired_ttest_q_BH",
        "wilcoxon_p",
        "wilcoxon_q_BH",
    ]

    for c in required_numeric_cols:
        if c not in df.columns:
            df[c] = np.nan

    summary = (
        df
        .groupby("Feature", as_index=False)
        .agg(
            Display_feature=("Display_feature", "first"),
            Ablation_appearance_count=("Feature", "count"),
            Methods_present=("Method", lambda x: ",".join(sorted(x.unique()))),
            Folds_present=("Outer_fold", lambda x: ",".join(map(str, sorted(x.unique())))),

            Mean_delta_F1=("delta_f1_mean(full-drop)", safe_mean),
            Median_delta_F1=("delta_f1_mean(full-drop)", safe_median),
            Mean_delta_F1_SD=("delta_f1_std", safe_mean),

            Mean_t_CI95_lower=("delta_f1_t_ci95_lower", safe_mean),
            Mean_t_CI95_upper=("delta_f1_t_ci95_upper", safe_mean),

            Mean_bootstrap_CI95_lower=("bootstrap_ci95_lower", safe_mean),
            Mean_bootstrap_CI95_upper=("bootstrap_ci95_upper", safe_mean),

            Mean_positive_split_ratio=("positive_split_ratio", safe_mean),

            Median_paired_ttest_p=("paired_ttest_p", safe_median),
            Median_paired_ttest_q_BH=("paired_ttest_q_BH", safe_median),
            Median_wilcoxon_p=("wilcoxon_p", safe_median),
            Median_wilcoxon_q_BH=("wilcoxon_q_BH", safe_median),

            Robust_count=("Decision", lambda x: int((x == "Robust positive contributor").sum())),
            Directional_count=("Decision", lambda x: int((x == "Directionally consistent contributor").sum())),
            Unstable_count=("Decision", lambda x: int((x == "Unstable or redundant").sum())),
        )
    )

    summary["Ablation_summary_decision"] = np.where(
        summary["Robust_count"] > 0,
        "At least one robust fold",
        np.where(
            summary["Directional_count"] > 0,
            "Directionally consistent in some folds",
            "Mostly unstable or redundant"
        )
    )

    summary = summary.sort_values(
        by=[
            "Robust_count",
            "Directional_count",
            "Mean_delta_F1",
            "Mean_positive_split_ratio"
        ],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    return df, summary


# ============================================================
# 5. Build candidate tables for manuscript use
# ============================================================

def build_candidate_tables(champion_summary, ablation_summary):
    merged = champion_summary.merge(
        ablation_summary,
        on=["Feature", "Display_feature"],
        how="left"
    )

    stable_features = merged[
        merged["Appearance_count"] >= MIN_APPEARANCES_FOR_PAPER
    ].copy()

    stable_ratio_candidates = stable_features[
        stable_features["Is_candidate_novel_ratio"] == True
    ].copy()

    core_stable = merged[
        merged["Stability_level"] == "Core stable feature"
    ].copy()

    stable_features = stable_features.sort_values(
        by=[
            "Appearance_count",
            "Mean_champion_score",
            "Mean_SHAP_importance"
        ],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    stable_ratio_candidates = stable_ratio_candidates.sort_values(
        by=[
            "Appearance_count",
            "Mean_champion_score",
            "Mean_SHAP_importance"
        ],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    core_stable = core_stable.sort_values(
        by=[
            "Appearance_count",
            "Mean_champion_score",
            "Mean_SHAP_importance"
        ],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return merged, stable_features, stable_ratio_candidates, core_stable


# ============================================================
# 6. Plotting functions
# ============================================================

def plot_top_features(summary_df, out_path, title, top_n=30):
    if summary_df.empty:
        return

    plot_df = summary_df.head(top_n).copy()
    plot_df = plot_df.sort_values("Appearance_count", ascending=True)

    plt.figure(figsize=(10, max(6, 0.32 * len(plot_df))), dpi=DPI)

    plt.barh(
        plot_df["Display_feature"].astype(str),
        plot_df["Appearance_count"].values
    )

    plt.xlabel("Appearance count across method-fold combinations", fontsize=12, fontweight="bold")
    plt.ylabel("Feature", fontsize=12, fontweight="bold")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlim(0, TOTAL_METHOD_FOLD_COMBOS)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()


def plot_score_vs_appearance(summary_df, out_path):
    if summary_df.empty:
        return

    plt.figure(figsize=(8, 6), dpi=DPI)

    x = summary_df["Appearance_count"].values
    y = summary_df["Mean_champion_score"].values

    plt.scatter(x, y, alpha=0.75)

    plt.xlabel("Appearance count", fontsize=12, fontweight="bold")
    plt.ylabel("Mean champion score", fontsize=12, fontweight="bold")
    plt.title("Champion Stability vs SHAP-Based Score", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()


# ============================================================
# 7. Main function
# ============================================================

def main():
    print("========== Reading rho=0.90 results ==========")

    all_champions, all_feature_scores, all_ablation = collect_all_rho090_results()

    if all_champions.empty:
        raise ValueError(
            "No ClusterChampions records were loaded. "
            "Please check the input path and file structure."
        )

    print(f"Champion record count: {len(all_champions)}")
    print(f"Feature-score record count: {len(all_feature_scores)}")
    print(f"Ablation record count: {len(all_ablation)}")

    print("\n========== Summarizing cluster-champion stability ==========")
    all_champions_raw, champion_summary = summarize_champion_stability(all_champions)

    print("\n========== Summarizing ablation statistics ==========")
    all_ablation_raw, ablation_summary = summarize_ablation(all_ablation)

    print("\n========== Building candidate tables ==========")
    merged_summary, stable_features, stable_ratio_candidates, core_stable = build_candidate_tables(
        champion_summary,
        ablation_summary
    )

    out_xlsx = OUT_DIR / "rho090_stable_champions_and_ratio_candidates_summary.xlsx"

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        all_champions_raw.to_excel(
            writer,
            sheet_name="all_champions_raw",
            index=False
        )

        champion_summary.to_excel(
            writer,
            sheet_name="champion_stability",
            index=False
        )

        if not all_feature_scores.empty:
            all_feature_scores.to_excel(
                writer,
                sheet_name="all_feature_scores_raw",
                index=False
            )

        if not all_ablation_raw.empty:
            all_ablation_raw.to_excel(
                writer,
                sheet_name="all_ablation_raw",
                index=False
            )

        if not ablation_summary.empty:
            ablation_summary.to_excel(
                writer,
                sheet_name="ablation_by_feature",
                index=False
            )

        merged_summary.to_excel(
            writer,
            sheet_name="merged_champion_ablation",
            index=False
        )

        stable_features.to_excel(
            writer,
            sheet_name="stable_feature_candidates",
            index=False
        )

        stable_ratio_candidates.to_excel(
            writer,
            sheet_name="stable_ratio_candidates",
            index=False
        )

        core_stable.to_excel(
            writer,
            sheet_name="core_stable_features",
            index=False
        )

    print(f"Summary Excel file saved to: {out_xlsx}")

    fig1 = OUT_DIR / "top_stable_champions_appearance_count.png"
    plot_top_features(
        champion_summary,
        fig1,
        title="Top Stable Cluster Champions",
        top_n=30
    )

    fig2 = OUT_DIR / "top_stable_ratio_candidates_appearance_count.png"
    plot_top_features(
        stable_ratio_candidates,
        fig2,
        title="Top Stable Candidate Ratio Features",
        top_n=30
    )

    fig3 = OUT_DIR / "champion_score_vs_appearance.png"
    plot_score_vs_appearance(
        champion_summary,
        fig3
    )

    print(f"Figure saved to: {fig1}")
    print(f"Figure saved to: {fig2}")
    print(f"Figure saved to: {fig3}")

    print("\n========== Top 20 core stable features ==========")
    show_cols = [
        "Display_feature",
        "Appearance_count",
        "Global_mean_count",
        "KNN_count",
        "Mean_champion_score",
        "Mean_SHAP_importance",
        "Mean_TopK_frequency",
        "Is_constructed_ratio",
        "Ratio_group",
        "Stability_level"
    ]
    show_cols = [c for c in show_cols if c in champion_summary.columns]
    print(champion_summary[show_cols].head(20))

    print("\n========== Top 20 stable candidate novel ratios ==========")
    show_cols2 = [
        "Display_feature",
        "Feature",
        "Appearance_count",
        "Global_mean_count",
        "KNN_count",
        "Mean_champion_score",
        "Mean_SHAP_importance",
        "Mean_TopK_frequency",
        "Ratio_group",
        "Stability_level",
        "Mean_delta_F1",
        "Mean_positive_split_ratio",
        "Ablation_summary_decision"
    ]
    show_cols2 = [c for c in show_cols2 if c in stable_ratio_candidates.columns]
    print(stable_ratio_candidates[show_cols2].head(20))

    print("\nAll tasks completed.")
    print(f"Output directory: {OUT_DIR}")


# ============================================================
# 8. Program entry point
# ============================================================

if __name__ == "__main__":
    main()

========== Reading rho=0.90 results ==========


FileNotFoundError: No rho=0.90 result file was found in: D:\桌面\Discriminants to High-Dimensional Ratio Data An Explainable Machine Learning Study of Granite Classification\Result\04_Foldwise_Correlation_Clustering_Champion_Selection_Rho_Sensitivity\global_mean\outer_fold_01\rho_0.90